# Programmable rotation gate array

Adapted from Qualtran v0.7.0, Copyright 2023 Google LLC, Apache-2.0.

In [ ]:
from collections import Counter
from IPython.display import SVG, display
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.compiler._component_trace.resource_model import estimate_component_resources
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.dynamic import DynamicCircuit

## `ProgrammableRotationGateArray`
An implementation of `ProgrammableRotationGateArrayBase` base class


This implementation of the `ProgrammableRotationGateArray` base class expects
all interleaved_unitaries to act on the `rotations_target` register.

See docstring of `ProgrammableRotationGateArrayBase` for more details.

In [ ]:
from examples.algorithms.programmable_rotation_array_catalog import ProgrammableRotationArrayExample

### Example Instances

In [ ]:
array = ProgrammableRotationArrayExample()

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("Programmable rotation array", array.signature, {})])))

### Call Graph

In [ ]:
graph = array.call_graph()
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
display(graph.leaf_counts)

### Compile the two lowering depths

The selection register chooses one of four angle words. The two-bit load word starts and ends at zero. The measured controlled rotations share the clean QROM scratch qubit and its classical slot after each lookup. The direct variant leaves three native controlled phases intact. The displayed source model counts the measured controlled-power decomposition.

In [ ]:
for op in (array, ProgrammableRotationArrayExample(measured_rotations=False)):
    circuit = DynamicCircuit(op.qubit_count)
    circuit.append(op)
    counts = Counter(gate.name for gate in circuit.to_ir().instructions)
    print("Emitted on", op.qubit_count, "wires", dict(counts))
    assert counts["measure"] == (9 if op.measured_rotations else 6)
print("Source logical resources", estimate_component_resources(array.resource_model()))